# Week 2 Corrections — Pipeline Completeness Audit

**Author:** Sharlet Maithya  
**Date:** April 2026  
**Purpose:** Corrected methodology for two findings from the Week 2 audit:
1. Silent recording false positive rate (corrected from point-in-time to historical)
2. Email 'failure' rate (corrected from delivery metric to engagement metric)

---

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Connection — credentials loaded from environment
# See docs/DS_GETTING_STARTED.md for setup instructions
DB_USER = os.environ['VETTED_DB_USER']
DB_PASS = os.environ['VETTED_DB_PASS']
DB_HOST = 'aws-1-eu-west-1.pooler.supabase.com'
DB_PORT = '5432'
DB_NAME = 'postgres'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)
print('Connected.')

---
## Finding 4 (Corrected): Silent Recording False Positive Rate

### What the original report said
> *"Our AI is 100% accurate — it has zero false positives on silent recordings."*

### Why it needed a correction

The original query measured **current state only** — it checked how many recordings are *currently* misclassified. 
But the platform had a documented incident in Feb 2026 where ~150 candidates were falsely scored zero 
due to Congrats' WPS heuristic misfiring on slow/hesitant speakers. A Gemini-based fallback verification 
system was built to recover them (`fn_verify_silent_transcription`).

The correct methodology uses `fallback_transcription_attempted_at` as a marker: 
- **Fallback attempted + still silent** → confirmed truly silent (Gemini agreed)
- **Fallback attempted + reclassified** → was a false positive, now recovered
- **No fallback attempted** → not yet checked (or submitted after the fix)

In [ ]:
silent_query = text("""
    SELECT
        COUNT(*) FILTER (WHERE transcription_status = 'silent'
            AND fallback_transcription_attempted_at IS NOT NULL)     AS confirmed_truly_silent,
        COUNT(*) FILTER (WHERE transcription_status != 'silent'
            AND fallback_transcription_attempted_at IS NOT NULL)     AS recovered_false_positives,
        COUNT(*) FILTER (WHERE transcription_status = 'silent'
            AND fallback_transcription_attempted_at IS NULL)         AS silent_not_yet_verified,
        COUNT(*)                                                     AS total_answers
    FROM candidate_answers
""")

with engine.connect() as conn:
    row = conn.execute(silent_query).fetchone()

confirmed       = row.confirmed_truly_silent
recovered       = row.recovered_false_positives
unverified      = row.silent_not_yet_verified
total           = row.total_answers
ever_silent     = confirmed + recovered  # total answers ever marked silent
fp_rate         = recovered / ever_silent * 100 if ever_silent > 0 else 0

print(f'Total candidate answers:                         {total:,}')
print(f'Ever marked silent (confirmed + recovered):      {ever_silent:,}')
print()
print(f'Confirmed truly silent (Gemini agreed):          {confirmed:,}')
print(f'Recovered false positives (Gemini reclassified): {recovered:,}')
print(f'Silent, not yet verified by Gemini:              {unverified:,}')
print()
print(f'Historical false positive rate:                  {fp_rate:.1f}%')
print(f'Currently unresolved false positives:            {unverified:,}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

categories = ['Confirmed\nTruly Silent', 'Recovered\nFalse Positives', 'Unverified\n(No Fallback Yet)']
values     = [confirmed, recovered, unverified]
colors     = ['#4ade80', '#f87171', '#94a3b8']

bars = ax.bar(categories, values, color=colors, width=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            str(val), ha='center', va='bottom', fontweight='bold')

ax.set_title('Silent Recording Audit — Historical View', fontsize=13, pad=12)
ax.set_ylabel('Number of Answers')
ax.set_ylim(0, max(values) * 1.2)

note = f'Historical false positive rate: {fp_rate:.1f}% of all silents ever flagged\nCurrent unresolved false positives: {unverified}'
ax.annotate(note, xy=(0.5, -0.22), xycoords='axes fraction',
            ha='center', fontsize=9, color='#475569')

plt.tight_layout()
plt.savefig('silent_recording_audit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: silent_recording_audit.png')

### Corrected Narrative

> The platform identified and recovered **{recovered} false-positive silent recordings** — {fp_rate:.0f}% of all answers ever marked silent. 
> After Gemini fallback verification, **zero unresolved false positives remain today**. 
> Detection accuracy improved significantly after a fix deployed in Feb 2026.

*(The original report's "0% false positives" was accurate for current state, but understated the historical scale of the issue and the recovery work done.)*

---
## Finding 2 (Corrected): Email 'Failure' Rate

### What the original report said
> *"Our email invites are failing 84% of the time."*

### Why it needed a correction

In Sendgrid terminology, **`sent` is not a failure** — it means Sendgrid accepted the email and is delivering it. 
Many mail servers don't send delivery receipts, so `sent` is a normal final state for successfully delivered email. 
True delivery failures are `bounced`, `dropped`, and `blocked`.

The 84% figure actually captured something real — but it's the **audition completion rate**, 
not a deliverability problem. Those are very different root causes and very different fixes.

In [ ]:
# Part A: Deliverability — how did Sendgrid classify our emails?
deliverability_query = text("""
    SELECT
        email_type,
        status,
        COUNT(*) AS count
    FROM email_delivery_log
    GROUP BY email_type, status
    ORDER BY email_type, count DESC
""")

with engine.connect() as conn:
    df_email = pd.DataFrame(conn.execute(deliverability_query).fetchall(),
                            columns=['email_type', 'status', 'count'])

# Overall deliverability
overall = df_email.groupby('status')['count'].sum().reset_index()
total_emails = overall['count'].sum()
print(f'Total emails logged: {total_emails}')
print()
print('Overall delivery status breakdown:')
for _, row in overall.sort_values('count', ascending=False).iterrows():
    label = '(FAILED)' if row['status'] in ('bounced','dropped','blocked') else ''
    print(f"  {row['status']:<12} {row['count']:>5}  ({row['count']/total_emails*100:.1f}%) {label}")

hard_failures = overall[overall['status'].isin(['bounced','dropped','blocked'])]['count'].sum()
print(f'\nTrue deliverability failure rate: {hard_failures}/{total_emails} = {hard_failures/total_emails*100:.1f}%')

In [ ]:
# Part B: Engagement — of candidates who received audition invites, how many completed?
engagement_query = text("""
    SELECT
        pc.status,
        COUNT(DISTINCT pc.id) AS candidates
    FROM email_delivery_log edl
    JOIN project_candidates pc ON pc.id = edl.project_candidate_id
    WHERE edl.email_type = 'audition_invite'
    GROUP BY pc.status
    ORDER BY candidates DESC
""")

with engine.connect() as conn:
    df_eng = pd.DataFrame(conn.execute(engagement_query).fetchall(),
                          columns=['status', 'candidates'])

total_invited = df_eng['candidates'].sum()
completed = df_eng[df_eng['status'].isin(['scored', 'unlocked'])]['candidates'].sum()
not_completed = total_invited - completed

print(f'Unique candidates who received an audition invite: {total_invited}')
print()
print('Status breakdown:')
for _, row in df_eng.iterrows():
    print(f"  {row['status']:<15} {row['candidates']:>4}  ({row['candidates']/total_invited*100:.1f}%)")

print(f'\nCompleted audition (scored/unlocked): {completed} ({completed/total_invited*100:.1f}%)')
print(f'Did not complete:                     {not_completed} ({not_completed/total_invited*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Deliverability
status_order = ['delivered', 'sent', 'bounced', 'dropped', 'blocked']
color_map = {
    'delivered': '#4ade80',
    'sent':      '#86efac',  # light green — not failed, just unconfirmed
    'bounced':   '#f87171',
    'dropped':   '#fca5a5',
    'blocked':   '#fcd34d',
}
inv_df = df_email[df_email['email_type'] == 'audition_invite'].groupby('status')['count'].sum()
inv_df = inv_df.reindex([s for s in status_order if s in inv_df.index])

axes[0].pie(inv_df.values,
            labels=[f"{s}\n({v})" for s, v in zip(inv_df.index, inv_df.values)],
            colors=[color_map[s] for s in inv_df.index],
            autopct='%1.1f%%', startangle=140)
axes[0].set_title('Audition Invite Deliverability\n(Sendgrid statuses)', fontsize=11)

sent_patch    = mpatches.Patch(color='#86efac', label='sent = accepted by Sendgrid (not a failure)')
axes[0].legend(handles=[sent_patch], loc='lower center', fontsize=8, framealpha=0.7)

# Right: Engagement funnel
funnel_labels = df_eng['status'].tolist()
funnel_vals   = df_eng['candidates'].tolist()
funnel_colors = ['#4ade80' if s in ('scored','unlocked') else '#94a3b8' for s in funnel_labels]

bars = axes[1].bar(funnel_labels, funnel_vals, color=funnel_colors)
for bar, val in zip(bars, funnel_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontweight='bold')

axes[1].set_title(f'Invite → Completion Engagement\n({completed}/{total_invited} completed = {completed/total_invited*100:.1f}%)',
                  fontsize=11)
axes[1].set_ylabel('Candidates')
axes[1].set_ylim(0, max(funnel_vals) * 1.2)
axes[1].tick_params(axis='x', rotation=20)

completed_patch = mpatches.Patch(color='#4ade80', label='Completed audition')
pending_patch   = mpatches.Patch(color='#94a3b8', label='Did not complete')
axes[1].legend(handles=[completed_patch, pending_patch], fontsize=9)

plt.suptitle('Email Analysis — Deliverability vs Engagement', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('email_analysis_corrected.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: email_analysis_corrected.png')

### Corrected Narrative

**Email deliverability is healthy.** True failures (bounced + dropped + blocked) account for <2% of all emails sent. 
Sendgrid `sent` status means the email was accepted and delivered — it is not a failure state.

**The real finding is post-delivery engagement.** Of 164 candidates who received an audition invite, 
only 23.8% completed the audition. 59.8% remain in `invited` status (never started), and 15.2% expired.

This is an **engagement/product problem**, not an email infrastructure problem. 
The fix belongs in email copy, timing, reminder cadence, or audition UX — not Sendgrid configuration.

---
*Original report language to update:*
> ~~"Our email invites are failing 84% of the time."~~

*Corrected language:*
> "Email deliverability is healthy (<2% hard failure rate). However, only 24% of candidates who receive an audition invite complete the audition — 60% never start, and 15% expire before doing so. This is an engagement gap, not a delivery problem."